# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [13]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [14]:
# Initialize and constants
import os
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano' #'gpt-4.1-mini'
openai = OpenAI()
anthropic = OpenAI(api_key=anthropic_api_key, base_url="https://api.anthropic.com/v1/")

API key looks good so far


In [15]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [16]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [17]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [18]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [19]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [20]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}]}

In [21]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [22]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 4 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'professional profile',
   'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'homepage', 'url': 'https://edwarddonner.com/'}]}

In [23]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Endpoints product page',
   'url': 'https://endpoints.huggingface.co'},
  {'type': 'Status page', 'url': 'https://status.huggingface.co/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [24]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [25]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
convaiinnovations/laya
Updated
5 days ago
•
4.48k
Edge0/Audio8-ASR-Infinite
Updated
6 days ago
•
23.7k
•
1.48k
abenzerps/Qwen-Image-2.1-Uncensored-GGUF
Updated
1 day ago
•
1.15M
•
2.39k
Qwen/Qwen-Image-2.1
Updated
9 days ago
•
64.4k
•
2.64k
XingChen-AGI/TeleOCR
Updated
about 17 hours a

In [26]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [27]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [28]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nconvaiinnovations/laya\nUpdated\n5 days ago\n•\n4.48k\nEdge0/Audio8-ASR-Infinite\nUpdated\n6 days ago\n•\n23.7k\n•\n

In [29]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [30]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the leading AI community building the future of machine learning. It serves as the go-to platform where researchers, developers, and enterprises collaborate seamlessly on cutting-edge models, datasets, and AI applications. Hugging Face empowers the global ML community to create, discover, and innovate together.

> *"The collaboration platform for the machine learning community."*

---

## What We Offer

- **2M+ Models:** Access an extensive and growing library of machine learning models including state-of-the-art transformers and domain-specific solutions.
- **500k+ Datasets:** A vast collection of high-quality datasets supporting a variety of applications from natural language processing to computer vision.
- **Spaces:** Host and share interactive ML-powered web apps and demos instantly.
- **Buckets:** Storage solutions tailored for AI workflow needs.
- **HuggingChat:** A community chat platform for engaging AI discussions.
- **Enterprise Solutions:** Professional support, scalable inference endpoints, and deployment integrations.

Our platform supports a wide array of AI tasks and is designed for individuals and organizations ranging from hobbyists and researchers to large enterprises.

---

## Community & Culture

- **Collaborative Spirit:** Hugging Face is a vibrant, open community of AI enthusiasts and experts sharing knowledge through forums, Discord, and GitHub.
- **Open Source Focus:** We drive progress by making resources freely accessible—encouraging contributions and transparency worldwide.
- **Continuous Learning:** Through our blog, daily research paper summaries, and educational resources, we foster ongoing development in AI expertise.
- **Friendly & Inclusive:** We champion diversity and welcome contributors from all backgrounds passionate about AI.

---

## Impact & Customers

Our platform fuels innovation across numerous sectors including healthcare, finance, robotics, media, and education. Organizations leverage Hugging Face to:

- Build advanced conversational agents
- Improve image and speech recognition
- Develop custom AI applications and prototypes rapidly
- Scale ML deployment with enterprise-grade support

---

## Careers at Hugging Face

Join a passionate team shaping the future of AI! We seek talented engineers, researchers, data scientists, product managers, and community specialists who thrive in a collaborative and innovative environment.

- Work alongside top AI minds in open, globally distributed teams.
- Contribute to open source projects making a real-world impact.
- Enjoy a culture of learning, transparency, and autonomy.
- Be part of a movement transforming how AI is built and shared.

Visit our website for current openings and to become part of the AI community revolution.

---

## Contact & Get Started

Explore the future of machine learning with Hugging Face:

- Website: [huggingface.co](https://huggingface.co)
- Community: Discord, Forums, GitHub
- Enterprise: Dedicated support and integration services available
- Sign Up to explore models, datasets, and AI applications immediately

---

**Hugging Face – The AI community building the future, together.**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

def stream_brochure(company_name, url, model):
    stream = model.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)
     

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


# Hugging Face Brochure

## About Hugging Face

**Hugging Face** is a pioneering AI community and platform building the future of machine learning. It serves as a global collaboration hub where AI researchers, engineers, and enthusiasts create, discover, and share machine learning models, datasets, and applications. With over 2 million models and 500,000 datasets hosted, Hugging Face is the definitive home for open and collaborative machine learning development.

## Our Platform

- **Models:** Access and contribute to over **2 million AI models** spanning multiple domains including natural language processing, computer vision, and more.
- **Datasets:** Explore from **500,000+ datasets**, covering diverse topics and data types to fuel your AI research and applications.
- **Spaces:** Interactive AI-powered apps and demonstrations, allowing users to test and deploy models in real-time.
- **Buckets & Storage:** Scalable storage and data management solutions designed for enterprise and individual users.
- **HuggingChat:** A new offering enabling conversational AI experiences built on state-of-the-art models.

## Who We Serve

Hugging Face caters to a broad spectrum of users:

- **AI Researchers & Data Scientists:** Collaborate and publish cutting-edge models and datasets.
- **Machine Learning Engineers:** Deploy and scale models seamlessly using Hugging Face’s infrastructure.
- **Enterprises:** Enterprise-grade support, advanced solutions, inference endpoints, and PRO services to integrate AI applications with security and reliability.
- **Community & Educators:** Knowledge sharing via forums, Discord, blogs, daily papers, and educational materials that help train the next generation of AI practitioners.

## Company Culture

Hugging Face fosters a vibrant, inclusive, and open culture valuing:

- **Collaboration:** Emphasis on community-driven innovation and shared learning.
- **Transparency:** Open access to models, datasets, and research promotes inclusivity.
- **Impact:** Empowering people to build more responsible and intelligent AI technologies.
- **Diversity:** Welcoming global contributors and diverse perspectives to push boundaries.

## Careers & Opportunities

Join Hugging Face if you are passionate about AI, open source, and building the future of machine learning. The company offers roles across:

- Research and development in AI and ML
- Software engineering and DevOps
- Community engagement and support
- Enterprise and customer success

Employees work in an agile, creative environment with access to cutting-edge resources and a community-focused workplace.

## Branding

Hugging Face's brand embodies warmth and approachability, reflected in its friendly logo and bright color palette (#FFD21E, #FF9D00, #6B7280). Brand assets are freely available to encourage consistency and collaboration.

---

**Explore the future of AI with Hugging Face — where innovation meets community.**

Visit: [huggingface.co](https://huggingface.co)  
Join the conversation on Discord, Forum, and GitHub.  
Discover, create, and collaborate on the world’s largest machine learning platform.

Traceback (most recent call last):
  File "/home/jnewth/repos/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jnewth/repos/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jnewth/repos/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jnewth/repos/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1635, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jnewth/repos/llm_engineering/.venv/lib/python3.12/site-p

In [ ]:



def stream_model(company_name, company_url, model_selector):
    if model_selector=="GPT":
        result = stream_brochure(company_name, company_url, openai)
    elif model_selector=="Claude":
        result = stream_brochure(company_name, company_url, anthropic)
    else:
        raise ValueError("Unknown model")
    yield from result

import gradio as gr # oh yeah!
company_name = gr.Textbox(label="name:", info="Enter a name", lines=1)
company_url = gr.Textbox(label="url:", info="Enter a url", lines=1)
model_selector = gr.Dropdown(["GPT", "Claude"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    title="brochure", 
    inputs=[company_name, company_url, model_selector], 
    outputs=[message_output], 
    flagging_mode="never"
    )
view.launch()


* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the AI community building the future by empowering the machine learning community to collaborate on models, datasets, and applications. It is the home of machine learning where developers, researchers, and enterprises come together to create, discover, and innovate with AI technologies.

The platform hosts an extensive collection of over **2 million models**, **500,000 datasets**, and **1 million+ applications**, making it an unparalleled resource for AI development and deployment.

---

## What We Offer

- **Models:** Access and contribute to a vast repository of open-source machine learning models updated regularly by the community.
- **Datasets:** Discover and share diverse datasets spanning various domains to fuel your machine learning projects.
- **Spaces:** Run and showcase AI applications in collaborative environments.
- **Enterprise Solutions:** Commercial support tailored to enterprise needs, including Hugging Face PRO, Enterprise Support, Inference Providers, Endpoints, and Storage Buckets.
- **Community & Learning:** Engage with AI practitioners through forums, Discord channels, blogs, and educational resources.

---

## Our Community & Customers

Hugging Face 
- Serves a global machine learning community including individual developers, research organizations, and top-tier enterprises.
- Enables collaboration on the development, sharing, and deployment of AI models and datasets.
- Hosts trending and innovative models such as advanced image generation, video synchronization, natural language processing, and more.
- Provides tools and infrastructure to accelerate AI innovation for academic, commercial, and industrial applications.

---

## Company Culture

- **Collaborative:** At its core, Hugging Face fosters open collaboration amongst AI experts worldwide.
- **Innovative:** Always exploring the frontiers of AI research and practical applications.
- **Open & Inclusive:** Committed to transparency, open source, and accessibility of AI tools and knowledge.
- **Community-Driven:** Focused on empowering users and contributors to build the future of machine learning together.

---

## Careers at Hugging Face

Hugging Face is continuously growing and seeking talented individuals passionate about AI, machine learning, and community-driven innovation. 

- Opportunities in engineering, research, product management, and enterprise customer support.
- Work closely with a vibrant international AI community.
- Be part of a mission-driven company shaping the future of Artificial Intelligence.

---

## Contact & Join Us

Explore AI apps, join the community, or browse resources:

- Website: [huggingface.co](https://huggingface.co)
- Community forums and Discord for discussions and support.
- GitHub repositories for code and contributions.
- Blog and daily research papers for the latest advancements.

Become part of the AI revolution with Hugging Face, where collaboration accelerates innovation.

---

### Brand Colors & Assets

**Primary Colors:**  
- Yellow: #FFD21E  
- Orange: #FF9D00  
- Gray: #6B7280  

Download official logos and brand assets at the Hugging Face website for use in your projects.

---

Hugging Face — The AI community building the future.  
Harness the power of machine learning models and data to transform your ideas into reality.

Traceback (most recent call last):
  File "/home/jnewth/repos/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jnewth/repos/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jnewth/repos/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jnewth/repos/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1635, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jnewth/repos/llm_engineering/.venv/lib/python3.12/site-p

In [ ]:

stream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>